In [ ]:
from pyspark.sql import SparkSession
# 1. Khởi tạo Spark Session kết nối đến Master chung của cụm
spark = SparkSession.builder \
    .appName("MetroPT3_Machine_Learning") \
    .master("spark://10.125.222.18:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()
# 2. Đường dẫn nạp dữ liệu sạch đã lưu trên HDFS
HDFS_ML = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
df = spark.read.parquet(HDFS_ML)
# 4. Kiểm tra cấu trúc dữ liệu để bắt đầu làm Classification
df.printSchema()
print(f"Tổng số bản ghi: {df.count()}")
df.createOrReplaceTempView('sensor')

In [ ]:
# Q6: PHÁT HIỆN BẤT THƯỜNG ÁP SUẤT THẤP KHI MÁY ĐANG VẬN HÀNH
q6_anomalies = spark.sql('''
    SELECT
        date,
        COUNT(*) AS so_lan_ap_thap
    FROM sensor
    WHERE COMP = 0
      AND Reservoirs < 8.0
    GROUP BY date
    ORDER BY so_lan_ap_thap DESC
    LIMIT 10
''')
print("\n========== EXECUTION PLAN ==========")
q6_anomalies.explain(True)
print("--- TOP 10 NGÀY CÓ ÁP SUẤT BÌNH CHỨA < 8 BAR ---")
df_q6_anomalies = q6_anomalies.toPandas()
try:
    display(df_q6_anomalies)
except NameError:
    print(df_q6_anomalies.to_string(index=False))